In [ ]:
# sanitize fused feats
import os, glob, torch

# in_dir  = r"C:\Users\Vivian\Documents\dsmil-wsi\temp_train_2_5"       # where current .pt files live
in_dir = r'C:\Users\Vivian\Documents\dsmil-wsi\temp_train_5_2'
out_dir = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52_patch224_fp\feats_pt1"  # cleaned outputs
os.makedirs(out_dir, exist_ok=True)

for pt in glob.glob(os.path.join(in_dir, "*.pt")):
    obj = torch.load(pt, map_location="cpu")

    if isinstance(obj, dict) and "features" in obj:
        X = obj["features"]
        # If someone previously concatenated labels, strip last column defensively
        if X.ndim == 2 and X.shape[1] > 1024 and (X.shape[1] % 1024) == 1:
            X = X[:, :-1]
        payload = {**obj, "features": X}
    else:
        # Plain tensor case: strip last column if needed and wrap as dict
        X = obj
        if X.ndim != 2:
            raise ValueError(f"Unexpected tensor shape in {pt}: {tuple(X.shape)}")
        # Heuristic: if fused dims = 1024 + 1024 + 1 (or other +1), drop last col
        if (X.shape[1] % 1024) == 1:
            X = X[:, :-1]
        payload = {"features": X}

    out_path = os.path.join(out_dir, os.path.basename(pt))
    torch.save(payload, out_path)
    print("Wrote:", out_path, "->", payload["features"].shape)


In [ ]:
# --- Batch: build Panther H5s by aligning with 2.5× features ---
import os, glob, h5py, torch, numpy as np
import pandas as pd
from numpy.linalg import norm

# ========= EDIT THESE PATHS =========
PT_DIR   = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52_patch224_fp\feats_pt1"  # sanitized .pt (dict with "features": [N,2048])
# H5_2P5   = r"C:\Users\Vivian\Documents\CLAM\CLAM\FEATURES_DIR_5x\FEATURES_DIR_2.5x\uniextracted_mag2x_patch224_fp\feats_h5" # 2.5 anchor
H5_2P5 = r'C:\Users\Vivian\Documents\PANTHER\PANTHER\features\uniextracted_mag5x_patch224_fp\feats_h5' # 5x anchor
OUT_DIR  = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52_patch224_fp\feats_h5"
D2       = 1024   # 2.5× feature dim in fused vector
SIM_THR  = 0.90   # warn if cosine < SIM_THR
os.makedirs(OUT_DIR, exist_ok=True)
# ====================================

def load_fused(pt_path):
    obj = torch.load(pt_path, map_location="cpu")
    X = obj["features"] if isinstance(obj, dict) and "features" in obj else obj
    X = np.asarray(X, dtype=np.float32)
    assert X.ndim == 2 and X.shape[1] == 2048, f"{pt_path}: expected [N,2048], got {X.shape}"
    return X

def load_2p5_h5(h5_path):
    with h5py.File(h5_path, "r") as f:
        coords = f["coords"][:]
        feats2 = f["features"][:] if "features" in f else None
        size_um = f["size_um"][:] if "size_um" in f else None
        if "mask" in f:
            m = f["mask"][:].astype(bool)
            coords = coords[m]
            if feats2 is not None and feats2.shape[0] == m.shape[0]:
                feats2 = feats2[m]
            if size_um is not None and size_um.shape[0] == m.shape[0]:
                size_um = size_um[m]
    return coords.astype(np.int32), (None if feats2 is None else feats2.astype(np.float32)), (None if size_um is None else size_um.astype(np.float32))

def cos_sim(A, B):
    A = A / np.clip(norm(A, axis=1, keepdims=True), 1e-12, None)
    B = B / np.clip(norm(B, axis=1, keepdims=True), 1e-12, None)
    return A @ B.T

def write_h5(out_path, feats, coords, size_um=None):
    with h5py.File(out_path, "w") as g:
        g.create_dataset("features", data=feats.astype(np.float32), dtype="float32", compression="gzip", compression_opts=4)
        g.create_dataset("coords",   data=coords.astype(np.int32),   dtype="int32",  compression="gzip", compression_opts=4)
        if size_um is not None:
            g.create_dataset("size_um", data=size_um.astype(np.float32), dtype="float32")
        g.attrs["feature_dim"] = int(feats.shape[1])
        g.attrs["schema"] = "panther_multiscale_v1_cosAligned"

logs = []

# iterate all sanitized .pt files
slides = [os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(PT_DIR, "*.pt"))]
for name in sorted(slides):
    pt_path = os.path.join(PT_DIR, f"{name}.pt")
    h5_path = os.path.join(H5_2P5, f"{name}.h5")
    out_path = os.path.join(OUT_DIR, f"{name}.h5")

    if not os.path.exists(h5_path):
        logs.append(dict(slide=name, status="missing_coord_h5"))
        continue

    X = load_fused(pt_path)                     # [Nf, 2048]
    coords, feats2p5, size_um = load_2p5_h5(h5_path)  # coords [Nc,2], feats2p5 [Nc,1024]
    Nf, Nc = X.shape[0], coords.shape[0]

    # if 2.5× features are present, align by cosine on the 2.5× block
    if feats2p5 is not None and feats2p5.shape[1] == D2:
        A = X[:, :D2]              # 2.5× block extracted from fused
        B = feats2p5               # original 2.5× features
        S = cos_sim(A, B)          # [Nf, Nc]
        best = S.argmax(axis=1)    # index of coord row per fused row
        best_sim = S.max(axis=1)

        # detect duplicates in mapping (rare; typically when Nc != Nf by 1)
        _, counts = np.unique(best, return_counts=True)
        dup = int((counts > 1).sum())

        # reorder coords (and size_um) to fused order
        coords_aligned = coords[best]
        size_um_aligned = (size_um[best] if size_um is not None else None)

        # write out (keep all fused rows; coord count will match Nf)
        write_h5(out_path, X, coords_aligned, size_um_aligned)

        logs.append(dict(
            slide=name, status="ok_cosalign",
            Nf=Nf, Nc=Nc,
            weak_matches=int((best_sim < SIM_THR).sum()),
            min_sim=float(best_sim.min()),
            max_sim=float(best_sim.max()),
            dup_coord_indexes=dup
        ))

    else:
        # Fallback: align by index and truncate to min length
        n = min(Nf, Nc)
        write_h5(out_path, X[:n], coords[:n], (None if size_um is None else size_um[:n]))
        logs.append(dict(
            slide=name, status="fallback_index_truncate",
            Nf=Nf, Nc=Nc, kept=n
        ))

# save a log for auditing
df = pd.DataFrame(logs).sort_values("slide")
display(df)
df.to_csv(os.path.join(OUT_DIR, "_build_log.csv"), index=False)

print(f"Done. Wrote H5s to: {OUT_DIR}")
print("Set Panther config in_dim = 2048")


,slide,status,Nf,Nc,weak_matches,min_sim,max_sim,dup_coord_indexes
0,FA 100 B1,ok_cosalign,3091,3092,3091,0.0,0.879082,110
1,FA 100 B2,ok_cosalign,3180,3181,3180,0.0,0.852132,132
2,FA 101 B1,ok_cosalign,2309,2310,2309,0.0,0.872691,101
3,FA 101 B2,ok_cosalign,2596,2597,2596,0.0,0.893421,116
4,FA 102 B1,ok_cosalign,2843,2844,2843,0.0,0.806125,131
...,...,...,...,...,...,...,...,...
235,PT 97 B2,ok_cosalign,1595,1596,1595,0.0,0.893428,56
236,PT 98 B1,ok_cosalign,2335,2336,2327,0.0,0.910838,101
237,PT 98 B2,ok_cosalign,2130,2131,2130,0.0,0.873484,89
238,PT 99 B1,ok_cosalign,2800,2801,2800,0.0,0.771573,92


Done. Wrote H5s to: C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52_patch224_fp\feats_h5
Set Panther config in_dim = 2048


Aligning by 5x feats

In [ ]:
# align_fused_pt_to_5x.py
import os, glob, h5py, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.optimize import linear_sum_assignment

# ====== EDIT THESE ======
FUSED_PT_DIR = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52x_patch224_fp\feats_pt1"          # {slide}.pt -> [Nf, 2048] or dict{"features": [Nf,2048]}
FIVE_H5_DIR  = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\features\uniextracted_mag5x_patch224_fp\feats_h5"       # {slide}.h5 with 'features' [Nc,1024], 'coords' [Nc,2]
OUT_DIR      = r"C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52x_patch224_fp\feats_h5_5x_aligned"
SIM_WARN     = 0.30                        # warn if matched cosine < this
# ========================

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def load_fused_pt(pt_path):
    obj = torch.load(pt_path, map_location="cpu")
    if isinstance(obj, dict) and "features" in obj:
        X = obj["features"]
    else:
        X = obj
    X = X.detach().cpu().numpy().astype(np.float32)
    assert X.ndim == 2 and X.shape[1] == 2048, f"{pt_path}: expected [N,2048], got {X.shape}"
    return X

def normed(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, 1e-12, None)

def align_one(slide):
    pt_path  = os.path.join(FUSED_PT_DIR, f"{slide}.pt")
    h5_path  = os.path.join(FIVE_H5_DIR,  f"{slide}.h5")
    out_h5   = os.path.join(OUT_DIR, f"{slide}.h5")
    out_map  = os.path.join(OUT_DIR, f"{slide}_match.csv")

    if not (os.path.exists(pt_path) and os.path.exists(h5_path)):
        print(f"[SKIP] {slide}: missing fused .pt or 5x .h5")
        return

    X = load_fused_pt(pt_path)  # [Nf, 2048]
    with h5py.File(h5_path, "r") as h:
        feats5  = h["features"][:].astype(np.float32)   # [Nc, 1024]
        coords5 = h["coords"][:].astype(np.int32)       # [Nc, 2]

    Nf, Nc = X.shape[0], feats5.shape[0]

    # Use the HIGH (5×) block of the fused vector
    A = normed(X[:, 1024:])   # [Nf,1024]
    B = normed(feats5)        # [Nc,1024]

    r = min(Nf, Nc)
    S = A[:r] @ B[:r].T       # cosine sim, [r,r]
    row_ind, col_ind = linear_sum_assignment(-S)  # maximize cosine

    sims = S[row_ind, col_ind]
    if sims.min() < SIM_WARN:
        print(f"[WARN] {slide}: low min cosine={sims.min():.3f} (mean={sims.mean():.3f})")

    # Reorder to 1-1 matched subset (length r)
    X_out      = X[row_ind]         # [r, 2048]
    coords_out = coords5[col_ind]   # [r, 2]

    # Write aligned H5 (no duplicate coords)
    with h5py.File(out_h5, "w") as g:
        g.create_dataset("features", data=X_out,  compression="gzip", compression_opts=4)
        g.create_dataset("coords",   data=coords_out, compression="gzip", compression_opts=4)
        g.attrs["feature_dim"] = int(X_out.shape[1])
        g.attrs["schema"]      = "aligned_by_5x_cos_hungarian"

    # Save mapping for audit
    pd.DataFrame({
        "fused_row": row_ind,
        "h5_row": col_ind,
        "cosine": sims
    }).to_csv(out_map, index=False)

    print(f"[OK] {slide}: Nf={Nf} Nc={Nc} -> kept r={r}, min/mean cos={sims.min():.3f}/{sims.mean():.3f} -> {out_h5}")

if __name__ == "__main__":
    slides = [os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(FIVE_H5_DIR, "*.h5"))]
    slides.sort()
    for s in slides:
        align_one(s)


[WARN] FA 100 B1: low min cosine=0.295 (mean=1.000)
[OK] FA 100 B1: Nf=3091 Nc=3092 -> kept r=3091, min/mean cos=0.295/1.000 -> C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52x_patch224_fp\feats_h5_5x_aligned\FA 100 B1.h5
[OK] FA 100 B2: Nf=3180 Nc=3181 -> kept r=3180, min/mean cos=0.321/1.000 -> C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52x_patch224_fp\feats_h5_5x_aligned\FA 100 B2.h5
[OK] FA 101 B1: Nf=2309 Nc=2310 -> kept r=2309, min/mean cos=0.492/1.000 -> C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52x_patch224_fp\feats_h5_5x_aligned\FA 101 B1.h5
[OK] FA 101 B2: Nf=2596 Nc=2597 -> kept r=2596, min/mean cos=0.649/1.000 -> C:\Users\Vivian\Documents\PANTHER\PANTHER\features\multiscale\52\uniextracted_mag52x_patch224_fp\feats_h5_5x_aligned\FA 101 B2.h5
[OK] FA 102 B1: Nf=2843 Nc=2844 -> kept r=2843, min/mean cos=0.418/1.000 -> C:\Users\Vivian\Documents\PANTHER\PANTHER\featur

: 